---
# 🟢 PARTIE EXTRACT

### Ex E1 ⭐ — Explorer les sources
Affichez pour chaque DataFrame : shape, colonnes, dtypes, et les 3 premières lignes.

In [3]:
import pandas as pd

produit = pd.read_csv("data/produits.csv")
client = pd.read_csv("data/clients.csv")
commande1 = pd.read_csv("data/commandes_Q1.csv")
commande2 = pd.read_csv("data/commandes_Q2.csv")

# creation d'un dataframe
dataframes = {
    "produit": produit,
    "client": client,
    "commande1": commande1,
    "commande2": commande2
}

for i, df in dataframes.items():
    # nombre de lignes et de colonnes
    print("Shape :", df.shape)

    # nom des colonnes
    print("Colonnes :", list(df.columns))

    # type des colonnes
    print("Types :\n", df.dtypes)

    # affiche les 3 premières lignes
    print("Aperçu :\n", df.head(3))
    
    print("\n")  # ligne vide pour séparer visuellement chaque DataFrame

Shape : (152, 7)
Colonnes : ['produit_id', 'nom', 'categorie', 'prix_unitaire', 'stock', 'fournisseur', 'actif']
Types :
 produit_id           str
nom                  str
categorie            str
prix_unitaire    float64
stock              int64
fournisseur          str
actif               bool
dtype: object
Aperçu :
   produit_id            nom     categorie  prix_unitaire  stock fournisseur  \
0    PRD0001  Produit_Inf_1  Informatique          57.96     71     TechPro   
1    PRD0002  Produit_Inf_2  Informatique         135.96    102   MediaShop   
2    PRD0003  Produit_Inf_3  Informatique          83.45    372    ElecPlus   

   actif  
0   True  
1   True  
2  False  


Shape : (515, 9)
Colonnes : ['client_id', 'nom', 'email', 'ville', 'region', 'segment', 'date_inscription', 'ca_historique', 'actif']
Types :
 client_id               str
nom                     str
email                   str
ville                   str
region                  str
segment                 str
date_

### Ex E2 ⭐ — Détecter les différences entre Q1 et Q2
Trouvez les colonnes présentes dans Q2 mais absentes de Q1, et vice-versa.
Affichez les différences clairement.

In [4]:
# TON CODE ICI
col_q1 = set(commande1.columns)
col_q2 = set(commande2.columns)

# colonnes présentes dans Q2 mais absentes de Q1
col_dans_q2 = col_q2 - col_q1

# colonnes présentes dans Q1 mais absentes de Q2
col_dans_q1 = col_q1 - col_q2

print("Colonnes présentes dans Q2 mais absentes de Q1 :", col_dans_q2)
print("Colonnes présentes dans Q1 mais absentes de Q2 :", col_dans_q1)

Colonnes présentes dans Q2 mais absentes de Q1 : {'moyen_paiement', 'devise'}
Colonnes présentes dans Q1 mais absentes de Q2 : {'mode_paiement'}


### Ex E3 ⭐ — Harmoniser les colonnes
1. Renommez la colonne problématique dans Q2 pour qu'elle corresponde à Q1
2. Ajoutez la colonne manquante dans Q1 avec une valeur par défaut
3. Vérifiez que les deux DataFrames ont maintenant les mêmes colonnes

In [5]:
# TON CODE ICI

#renommer la colonne moyen
commande2 = commande2.rename(columns={"moyen_paiement":"mode_paiement"})

#ajout de la colonne manquante
commande1["devise"] = "EUR"

#verification des deux df
print("Colonnes identiques :", set(commande1.columns) == set(commande2.columns))


Colonnes identiques : True


### Ex E4 ⭐ — Consolider avec pd.concat
1. Ajoutez une colonne `source` dans Q1 (`'Q1_2024'`) et Q2 (`'Q2_2024'`)
2. Concaténez Q1 et Q2 avec `pd.concat`
3. Vérifiez que le total de lignes est cohérent
4. Affichez la répartition des lignes par source

In [6]:
# TON CODE ICI
#ajout de colonne source
commande1["source"] = "Q1_2024"
commande2["source"] = "Q2_2024"

commandes = pd.concat([commande1, commande2])

print("Le total de lignes commande1 :", len(commande1))
print("Le total de lignes commande2 :", len(commande2))
print("Les lignes consolidées :", len(commandes))

print(commandes["source"].value_counts())

Le total de lignes commande1 : 3020
Le total de lignes commande2 : 3220
Les lignes consolidées : 6240
source
Q2_2024    3220
Q1_2024    3020
Name: count, dtype: int64


In [7]:
commandes.columns

Index(['commande_id', 'client_id', 'produit_id', 'quantite', 'prix_unitaire',
       'montant_total', 'date_commande', 'statut', 'mode_paiement',
       'region_livraison', 'devise', 'source'],
      dtype='str')

### Ex E5 ⭐⭐ — Fonction extract()
Encapsulez tout dans une fonction `extract(q1_path, q2_path, clients_path, produits_path) -> dict`
qui retourne un dictionnaire avec les 4 DataFrames harmonisés et consolidés.
La fonction doit afficher des logs à chaque étape.

In [30]:
import pandas as pd
import logging

# Configuration du logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger("ETL")


def extract(q1_path, q2_path, clients_path, produits_path) -> dict:
    """
    Charge les fichiers CSV, harmonise les colonnes,
    consolide les commandes et retourne les DataFrames.
    """

    logger.info("EXTRACT — Début")

    logger.info("Chargement des fichiers...")

    commandes1 = pd.read_csv(q1_path)
    commandes2 = pd.read_csv(q2_path)
    clients = pd.read_csv(clients_path)
    produits = pd.read_csv(produits_path)

    logger.info(clients.shape)
    logger.info(produits.shape)

    logger.info("Fichiers chargés avec succès.")

    logger.info("Harmonisation des colonnes...")

    # Harmonisation du nom de colonne
    commandes2 = commandes2.rename(
        columns={"moyen_paiement": "mode_paiement"}
    )

    # Ajout de la colonne manquante
    commandes1["devise"] = "EUR"

    logger.info("Colonnes harmonisées.")

    logger.info("Ajout de la colonne source...")

    commandes1["source"] = "Q1_2024"
    commandes2["source"] = "Q2_2024"

    logger.info("Concaténation des commandes...")

    commandes = pd.concat(
        [commandes1, commandes2],
        ignore_index=True
    )

    logger.info(
        f"Consolidation terminée : {len(commandes)} lignes."
    )

    logger.info("EXTRACT — Fin")

    return {
        "commandes": commandes,
        "clients": clients,
        "produits": produits
    }


# Appel de la fonction
data = extract(
    "data/commandes_Q1.csv",
    "data/commandes_Q2.csv",
    "data/clients.csv",
    "data/produits.csv"
)

# Affichage des premières lignes
data["commandes"].head()

14:58:41 | EXTRACT — Début
14:58:41 | Chargement des fichiers...
14:58:41 | (515, 9)
14:58:41 | (152, 7)
14:58:41 | Fichiers chargés avec succès.
14:58:41 | Harmonisation des colonnes...
14:58:41 | Colonnes harmonisées.
14:58:41 | Ajout de la colonne source...
14:58:41 | Concaténation des commandes...
14:58:41 | Consolidation terminée : 6240 lignes.
14:58:41 | EXTRACT — Fin


,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devise,source
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,EUR,Q1_2024
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,EUR,Q1_2024
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,EUR,Q1_2024
3,CMD-Q1-00004,CLI0381,PRD0061,8,75.83,606.64,2024-01-17,annulée,PayPal,PACA,EUR,Q1_2024
4,CMD-Q1-00005,CLI0187,PRD0040,4,71.91,287.64,2024-03-20,en_cours,CB,Occitanie,EUR,Q1_2024


---
# 🟡 PARTIE TRANSFORM

### Ex T1 ⭐ — Audit initial
Écris une fonction `audit_dataframe(df, nom)` qui affiche :
- Shape
- Nombre de NaN total et par colonne (en %)
- Nombre de doublons
- Types de chaque colonne

In [31]:
# TON CODE ICI
def audit_dataframe(df, nom):
    print(f"--- Audit de {nom} ---")

    print("shape :", df.shape)
    
    # isna() renvoie True/False pour chaque valeur manquante
    # .sum() compte le nombre total de True (donc de NaN) par colonne
    total_nan = df.isna().sum()

    # pourcentage de NaN par colonne
    pourcentage_nan = (total_nan / len(df)) * 100

    print("Nombre de NaN total :", total_nan.sum())
    print("NaN par colonne (%) :\n", pourcentage_nan)

    # nombre de doublons
    print(f"Nombre de doublons :, {df.duplicated().sum()}")
    
    # types de chaque colonne
    print("Types :\n", df.dtypes)

audit_dataframe(data["commandes"], "Commandes")

--- Audit de Commandes ---
shape : (6240, 12)
Nombre de NaN total : 179
NaN par colonne (%) :
 commande_id         0.00000
client_id           0.00000
produit_id          0.00000
quantite            0.00000
prix_unitaire       0.00000
montant_total       1.87500
date_commande       0.00000
statut              0.99359
mode_paiement       0.00000
region_livraison    0.00000
devise              0.00000
source              0.00000
dtype: float64
Nombre de doublons :, 40
Types :
 commande_id             str
client_id               str
produit_id              str
quantite              int64
prix_unitaire       float64
montant_total       float64
date_commande           str
statut                  str
mode_paiement           str
region_livraison        str
devise                  str
source                  str
dtype: object


### Ex T2 ⭐ — Supprimer les doublons
Sur le DataFrame consolidé :
1. Détectez et affichez le nombre de doublons
2. Montrez 3 exemples de lignes dupliquées
3. Supprimez les doublons (gardez la première occurrence)
4. Vérifiez qu'il n'en reste plus

In [32]:
# TON CODE ICI
commandes = data["commandes"]
#nombre de doublon
nb_doublon = commandes.duplicated().sum()
print(f"Nombre de doublons détectés : {nb_doublon}")


Nombre de doublons détectés : 40


In [33]:
# 3 exemples de lignes dupliquées
ligne_duplique = commandes[commandes.duplicated(keep=False)]
ligne_duplique.head(3)

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devise,source
13,CMD-Q1-00014,CLI0208,PRD0090,5,80.66,403.30,2024-02-05,livrée,CB,Nouvelle-Aquitaine,EUR,Q1_2024
71,CMD-Q1-00072,CLI0001,PRD0027,7,45.84,320.88,2024-03-20,en_cours,CB,Île-de-France,EUR,Q1_2024
144,CMD-Q1-00145,CLI0441,PRD0020,9,304.72,2742.48,2024-02-19,livrée,CB,Île-de-France,EUR,Q1_2024


In [34]:
# suppression des doublons en gardant la première
commandes = commandes.drop_duplicates(keep="first")

# verification
nb_doublon = commandes.duplicated().sum()
print(f"Doublons restants : {nb_doublon}")


Doublons restants : 0


### Ex T3 ⭐ — Corriger les types
1. Convertissez `date_commande` en datetime en gérant les formats incohérents
2. Extrayez : `annee`, `mois`, `trimestre`, `jour_semaine`
3. Convertissez `quantite` en numérique avec gestion des erreurs
4. Affichez les types avant et après

In [35]:
commandes.dtypes

commande_id             str
client_id               str
produit_id              str
quantite              int64
prix_unitaire       float64
montant_total       float64
date_commande           str
statut                  str
mode_paiement           str
region_livraison        str
devise                  str
source                  str
dtype: object

In [36]:
# pd.to_datetime convertit une colonne en vraies dates
# errors="coerce" transforme les dates mal formatées en NaT (Not a Time) au lieu de faire planter le code

commandes["date_commande"] = pd.to_datetime(commandes["date_commande"], errors="coerce")

# extraction des colonnes dates
# pd.to_datetime convertit une colonne en vraies dates
commandes["annee"] = commandes["date_commande"].dt.year
commandes["mois"] = commandes["date_commande"].dt.month
commandes["trimestre"] = commandes["date_commande"].dt.quarter
commandes["jour_semaine"] = commandes["date_commande"].dt.day_name(locale='fr_FR')

# pd.to_numeric convertit une colonne en nombre
commandes["quantite"] = pd.to_numeric(commandes["quantite"], errors="coerce")

commandes.dtypes

commande_id                    str
client_id                      str
produit_id                     str
quantite                     int64
prix_unitaire              float64
montant_total              float64
date_commande       datetime64[us]
statut                         str
mode_paiement                  str
region_livraison               str
devise                         str
source                         str
annee                      float64
mois                       float64
trimestre                  float64
jour_semaine                   str
dtype: object

In [37]:
commandes["mois"]

0       3.0
1       1.0
2       1.0
3       1.0
4       3.0
       ... 
6215    6.0
6216    6.0
6217    5.0
6218    5.0
6219    6.0
Name: mois, Length: 6200, dtype: float64

### Ex T4 ⭐⭐ — Traiter les NaN
Pour chaque colonne avec des NaN, choisissez et appliquez la stratégie adaptée :
- `montant_total` NaN → recalculer depuis `quantite × prix_unitaire`
- `statut` NaN → valeur par défaut `'inconnu'`
- `date_commande` NaT → supprimer la ligne (donnée critique)

Justifiez chaque choix dans un commentaire.

In [38]:
# TON CODE ICI
# on recalcule montant_total uniquement là où il est NaN

montant_nan = commandes["montant_total"].isna()
commandes.loc[montant_nan, "montant_total"] = (
    commandes.loc[montant_nan, "quantite"] * commandes.loc[montant_nan, "prix_unitaire"]
)

# justification : montant_total est une donnée dérivée d'autres colonnes, donc on peut la reconstruire au lieu de perdre la ligne

# vérifier qu'il ne reste plus de NaN sur montant_total
print("NaN restants sur montant_total :", commandes["montant_total"].isna().sum())

NaN restants sur montant_total : 0


In [39]:
# remplacer les NaN de "statut" par "inconnu"
# fillna() remplace toutes les valeurs manquantes d'une colonne par une valeur donnée
commandes["statut"] = commandes["statut"].fillna("inconnu")

# justification : le statut est une info utile mais pas critique, on préfère garder la ligne avec une valeur neutre plutôt que la supprimer

# vérifier qu'il ne reste plus de NaN sur statut
print("NaN restants sur statut :", commandes["statut"].isna().sum())

NaN restants sur statut : 0


In [40]:
# suppression des lignes où date_commande est NaT (manquante)
# dropna() supprime les lignes qui ont un NaN dans la colonne indiquée (subset=[...])
commandes = commandes.dropna(subset=["date_commande"])

# justification : sans date, on peut pas savoir à quel trimestre ou mois, rattacher la commande
# c'est une donnée critique pour l'analyse temporelle, donc on préfère supprimer la ligne plutôt que de fausser les résultats

commandes["date_commande"].isna().sum()

np.int64(0)

### Ex T5 ⭐⭐ — Détecter et traiter les aberrants
1. Détectez les prix négatifs → supprimez ces lignes
2. Détectez les quantités nulles ou négatives → supprimez
3. Vérifiez la cohérence `montant_total = quantite × prix_unitaire` → corrigez les incohérences
4. Affichez un résumé des corrections effectuées

In [41]:
# 1. detection et suppression des prix négatifs
nb_prix_negatif = (commandes["prix_unitaire"] < 0).sum()
# on garde uniquement les lignes où le prix est positif ou nul
commandes = commandes[commandes["prix_unitaire"] >= 0]

# 2. detection et suppression des quantités nulles ou négatives
nb_qte_negative = (commandes["quantite"] <= 0).sum()
#ligne où la quantité est non nulle
commandes = commandes[commandes["quantite"] > 0]

# 3. Vérification et correction d'incohérence (montant_total = quantite × prix_unitaire)
nvo_montant = (commandes["quantite"] * commandes["prix_unitaire"]).round(2)   #evite les faux positifs
incoherence = (commandes["montant_total"].round(2) != nvo_montant )

# correction : on remplace montant_total par la valeur recalculée
# uniquement sur les lignes incohérentes
commandes.loc[incoherence, "montant_total"] = nvo_montant[incoherence]

print(f"Prix négatifs détectés et suprimés: {nb_prix_negatif}")
print(f"Quantités nulles ou négatives détectées et supprimées : {nb_qte_negative}")
print(f"Nombre d'incohérences montant_total corrigé: {incoherence.sum()}")

Prix négatifs détectés et suprimés: 22
Quantités nulles ou négatives détectées et supprimées : 29
Nombre d'incohérences montant_total corrigé: 0


### Ex T6 ⭐⭐ — Jointures avec les référentiels
1. Nettoyez `df_clients` (doublons sur `client_id`)
2. Filtrez `df_produits` pour ne garder que les produits actifs
3. Faites un LEFT JOIN commandes ← clients (colonnes : ville, region, segment)
4. Faites un LEFT JOIN commandes ← produits (colonnes : nom, categorie, fournisseur)
clients = data["client"]5. Vérifiez le nombre de lignes avant et après (doit être identique avec LEFT JOIN)

In [42]:
clients = data["clients"]
produits = data["produits"]

# nombre de lignes avant les jointures
nb_lignes_avant = len(commandes)
print("Lignes avant les jointures :", nb_lignes_avant)

# 1.nettoyer df_clients et supprimer les doublons
clients = clients.drop_duplicates(subset=["client_id"], keep="first")

# 2.filtrer df_produits pour ne garder que les produits actifs
produits_actifs = produits[produits["actif"] == True]

# 3.left join commandes <- clients
commandes = commandes.merge(
    clients[[ "client_id", "ville", "region", "segment" ]],
    on = "client_id",
    how = "left"
)
#-----------------------------------------------------------------
# nombre de lignes après les jointures
nb_lignes_apres = len(commandes)
print("Lignes après les jointures :", nb_lignes_apres)

# 4.left join commandes <- produits
commandes = commandes.merge(
    produits_actifs[["produit_id", "nom", "categorie", "fournisseur" ]],
    on = "produit_id",
    how = "left"
)
#------------------------------------------------------------------
# nombre de lignes après les jointures
nb_lignes_apres = len(commandes)
print("Lignes après les jointures :", nb_lignes_apres)

# 5. verification
if nb_lignes_avant == nb_lignes_apres:
    print("Le nombre de lignes est conservé après les jointures.")
else:
    print("Le nombre de lignes a changé.")

Lignes avant les jointures : 6095
Lignes après les jointures : 6095
Lignes après les jointures : 6095
Le nombre de lignes est conservé après les jointures.


### Ex T7 ⭐⭐ — Feature Engineering
Créez les colonnes suivantes :
1. `tranche_montant` : `<100€` / `100-500€` / `500-2000€` / `>2000€` avec `pd.cut()`
2. `saison` : depuis le mois (Hiver/Printemps/Été/Automne)
3. `est_annulee` : booléen True si statut == 'annulée'
4. `ca_perdu` : montant_total si commande annulée, sinon 0
5. `delai_inscription_commande` : nb de jours entre date_inscription du client et date_commande

Affichez `value_counts()` pour les colonnes catégorielles créées.

In [43]:
# 1.colonne tranche_montant
# pd.cut() coupe une colonne numérique en catégories selon des bornes
# bins définit les limites, labels donne le nom de chaque tranche
commandes["tranche_montant"] = pd.cut(
    commandes["montant_total"],
    bins=[0, 100, 500, 2000, float("inf")],
    labels=["<100€", "100-500€", "500-2000€", ">2000€"]
)

# 2.colonne saison
def saison(mois):
    if mois in [12, 1, 2]:
        return "Hiver"
    elif mois in [3, 4, 5]:
        return "Printemps"
    elif mois in [6, 7, 8]:
        return "Été"
    else:
        return "Automne"

commandes["saison"] = commandes["mois"].apply(saison)

# 3.colonne est_annulee : booléen True si statut == 'annulée'
commandes["est_annulee"] = commandes["statut"] == "annulée"

# 4.colonne ca_perdu : montant_total si commande annulée, sinon 0
commandes["ca_perdu"] = commandes["montant_total"].where(commandes["est_annulee"], 0)

# 5. colonne delai_inscription_commande : nb de jours entre date_inscription du client et date_commande
commandes = commandes.merge(
    clients[["client_id", "date_inscription"]],
    on="client_id",
    how="left"
)

# colonne date_inscription
commandes["date_inscription"] = pd.to_datetime(commandes["date_inscription"], errors="coerce")

# la soustraction entre deux dates donne un "Timedelta"
# .dt.days permet d'extraire uniquement le nombre de jours (sous forme de nombre entier)
commandes["delai_inscription_commande"] = (
    commandes["date_commande"] - commandes["date_inscription"]
).dt.days

# affichage
print(commandes["tranche_montant"].value_counts())
print(commandes["saison"].value_counts())
print(commandes["est_annulee"].value_counts())

tranche_montant
100-500€     2933
500-2000€    2147
<100€         636
>2000€        379
Name: count, dtype: int64
saison
Printemps    3063
Hiver        1990
Été          1042
Name: count, dtype: int64
est_annulee
False    5493
True      602
Name: count, dtype: int64


### Ex T8 ⭐⭐⭐ — Validation qualité
Écris une fonction `valider_pipeline(df) -> bool` qui :
1. Vérifie qu'il n'y a plus de NaN sur les colonnes critiques
2. Vérifie qu'il n'y a plus de doublons
3. Vérifie que tous les prix sont > 0
4. Vérifie que toutes les quantités sont > 0
5. Vérifie que montant_total == quantite × prix_unitaire (à 1 centime près)
6. Retourne True si tout est OK, False sinon avec les erreurs détaillées

In [44]:
def valider_pipeline(df) -> bool :
    
    problemes = []  # stocke la liste des problèmes trouvés

    # 1. Vérifier qu'il n'y a plus de NaN sur les colonnes critiques
    col_critique = ["montant_total", "quantite", "prix_unitaire", "date_commande"]
    for i in col_critique :
        nb_nan = df[i].isna().sum()
        if nb_nan > 0 :
            problemes.append(f"{nb_nan} valeurs manquantes dans la colonne '{i}'")

    # 2. Vérifier qu'il n'y a plus de doublons
    nb_doublon = df.duplicated().sum()
    if nb_doublon > 0:
        problemes.append(f"{nb_doublon} doublons détectés")

    # 3. Vérifier que tous les prix sont > 0
    nb_prix_invalides = (df["prix_unitaire"] <= 0).sum()
    if nb_prix_invalides > 0:
        problemes.append(f"{nb_prix_invalides} lignes avec prix_unitaire <= 0")

    # 4. Vérifier que toutes les quantités sont > 0
    nb_qte_invalides = (df["quantite"] <= 0).sum()
    if nb_qte_invalides > 0:
        problemes.append(f"{nb_qte_invalides} lignes avec quantite <= 0")

    # 5. Vérifier que montant_total == quantite × prix_unitaire (à 1 centime près)
    nvo_montant = (df["quantite"] * df["prix_unitaire"]).round(2)
    nb_incoherence = (df["montant_total"].round(2) != nvo_montant).sum()
    if nb_incoherence > 0:
        problemes.append(f" il existe {nb_incoherence} lignes où montant_total ne correspond pas à quantite * prix_unitaire")

    # 6. Retourne True si tout est OK, False sinon avec les erreurs détaillées
    if len(problemes) == 0:
        print(" Aucune erreur détectée. Validation réussie !")
        return True
    else:
        print("Echec :")
        for j in problemes:
            print(j)
        return False


valider_pipeline(commandes)

 Aucune erreur détectée. Validation réussie !


True

### Ex T9 ⭐⭐⭐ — Analyse métier post-transform
Maintenant que les données sont propres, répondez en code :
1. Quel est le CA total du S1 2024 ?
2. Quel trimestre a le CA le plus élevé ?
3. Quelle région génère le plus de CA ?
4. Quel segment client est le plus rentable ?
5. Quelle catégorie de produit a le taux d'annulation le plus élevé ?
6. Quel est le CA perdu sur les annulations ?

In [23]:
commandes.head()

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,...,segment,nom,categorie,fournisseur,tranche_montant,saison,est_annulee,ca_perdu,date_inscription,delai_inscription_commande
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,...,Particulier,NaN,NaN,NaN,<100€,Printemps,True,49.36,2021-07-26,966.0
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,...,Startup,Produit_Aud_58,Audio,TechPro,500-2000€,Hiver,False,0.00,2023-03-31,298.0
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,...,Particulier,NaN,NaN,NaN,500-2000€,Hiver,False,0.00,2020-07-12,1282.0
3,CMD-Q1-00004,CLI0381,PRD0061,8,75.83,606.64,2024-01-17,annulée,PayPal,PACA,...,Particulier,Produit_Aud_61,Audio,DataStore,500-2000€,Hiver,True,606.64,2023-01-07,375.0
4,CMD-Q1-00005,CLI0187,PRD0040,4,71.91,287.64,2024-03-20,en_cours,CB,Occitanie,...,Particulier,NaN,NaN,NaN,100-500€,Printemps,False,0.00,2020-10-24,1243.0


In [45]:
# 1. CA total du S1 2024
ca_s1_2024 = commandes[
    (commandes["annee"] == 2024) & (commandes["mois"] <= 6)
    ]["montant_total"].sum()
print(f"CA total S1 2024 : {ca_s1_2024}")

# 2. quel trimestre a le CA le plus élevé ?
ca_par_trimestre = commandes.groupby("trimestre")["montant_total"].sum()
print(f"CA par trimestre :\n", ca_par_trimestre)
print(f"Trimestre avec le CA le plus élevé : {ca_par_trimestre.idxmax()}")  # idxmax = l'index c.à.d le nom du groupe qui a la valeur max

# 3. quelle région génère le plus de CA ?
ca_par_region = commandes.groupby("region")["montant_total"].sum()
print(f"Région avec le plus de CA : {ca_par_region.idxmax()}")

# 4. quel segment client est le plus rentable ?
ca_par_segment = commandes.groupby("segment")["montant_total"].sum()
print(f"Segment le plus rentable : {ca_par_segment.idxmax()}")

# 5. quelle catégorie de produit a le taux d'annulation le plus élevé ?
taux_annulation_par_categorie = commandes.groupby("categorie")["est_annulee"].mean() # pour obtenir un taux
print("Taux d'annulation par catégorie :\n", taux_annulation_par_categorie)
print(f"Catégorie avec le taux d'annulation le plus élevé : {taux_annulation_par_categorie.idxmax()}")

# 6. CA perdu sur les annulations
ca_perdu = commandes["ca_perdu"].sum()
print(f"CA perdu total sur les annulations : {ca_perdu}")


CA total S1 2024 : 3996151.3100000005
CA par trimestre :
 trimestre
1.0    1920303.91
2.0    2075847.40
Name: montant_total, dtype: float64
Trimestre avec le CA le plus élevé : 2.0
Région avec le plus de CA : Nouvelle-Aquitaine
Segment le plus rentable : Particulier
Taux d'annulation par catégorie :
 categorie
Accessoires     0.100352
Audio           0.107078
Informatique    0.084770
Réseau          0.117647
Stockage        0.088634
Téléphonie      0.099462
Name: est_annulee, dtype: float64
Catégorie avec le taux d'annulation le plus élevé : Réseau
CA perdu total sur les annulations : 406786.26


---
# 🔵 PARTIE LOAD

### Ex L1 ⭐ — Export CSV
1. Exportez le DataFrame enrichi en CSV dans `../output/commandes_enrichies.csv`
2. Vérifiez la taille du fichier en Ko
3. Rechargez le CSV et vérifiez que le shape est identique

In [46]:
import os

# 1. exportation du DataFrame en CSV

# on crée le dossier "output" s'il n'existe pas déjà
os.makedirs("output", exist_ok=True)

# index=False évite d'ajouter une colonne "index" inutile dans le fichier exporté
commandes.to_csv("output/commandes_enrichies.csv", index=False)

# 2. vérification de la taille du fichier en Ko
taille_ko = os.path.getsize("output/commandes_enrichies.csv") / 1024  # os.path.getsize() renvoie la taille en octets
print(f"Taille du fichier CSV : {taille_ko:.2f} Ko") # le ':.2f' est une syntaxe de formatage de nombre dans une f-string 

# 3. recharger le CSV pour vérifier qu'il n'y a pas eu de perte de données
csv_df = pd.read_csv("output/commandes_enrichies.csv")

# verification du shape avant et après export/rechargement
print("Shape original :", commandes.shape)
print("Shape rechargé :", csv_df.shape)
print("Shapes identiques :", commandes.shape == csv_df.shape)


Taille du fichier CSV : 1297.99 Ko
Shape original : (6095, 28)
Shape rechargé : (6095, 28)
Shapes identiques : True


### Ex L2 ⭐⭐ — Export Parquet
1. Exportez en Parquet dans `../output/commandes_S1.parquet`
2. Comparez la taille CSV vs Parquet
3. Rechargez le Parquet et vérifiez le shape
4. Mesurez le temps de lecture CSV vs Parquet avec `%%time`

In [47]:
import os
import time

# 1. exportez en parquet
commandes.to_parquet("output/commandes_S1.parquet", index = False)
taille_parquet =  os.path.getsize("output/commandes_S1.parquet") / 1024
taille_csv = os.path.getsize("output/commandes_enrichies.csv") / 1024

# 2. comparez la taille CSV vs Parquet
print(f"Taille du fichier Parquet : {taille_parquet:.2f} Ko")
print(f"Taille du fichier CSV : {taille_csv:.2f} Ko")
print(f"Ratio de compression : {taille_csv / taille_parquet:.2f}x plus petit en Parquet")

# 3. rechargez le Parquet et vérifiez le shape
parquet_df = pd.read_parquet("output/commandes_S1.parquet")

# verification du shape avant et après export/rechargement
print("Shape original :", commandes.shape)
print("Shape rechargé :", parquet_df.shape)
print("Shapes identiques :", commandes.shape == parquet_df.shape)


Taille du fichier Parquet : 163.36 Ko
Taille du fichier CSV : 1297.99 Ko
Ratio de compression : 7.95x plus petit en Parquet
Shape original : (6095, 28)
Shape rechargé : (6095, 28)
Shapes identiques : True


In [48]:
# 4.a Mesurez le temps de lecture CSV 
start = time.perf_counter() #stock le temps au début du chargement du fichier
lecture_csv = pd.read_csv("output/commandes_enrichies.csv")
end = time.perf_counter() #stock le temps à la fin du chargement
print(f"Lecture CSV : {end - start:.4f} secondes")

# 4.b Mesurez le temps de lecture Parquet
start = time.perf_counter()
lecture_parquet = pd.read_parquet("output/commandes_S1.parquet")
end = time.perf_counter()
print(f"Lecture parquet : {end - start:.4f} secondes")

Lecture CSV : 0.0773 secondes
Lecture parquet : 0.0136 secondes


### Ex L3 ⭐⭐ — SQLite
1. Créez une base SQLite `../output/ecommerce.db`
2. Chargez le DataFrame dans une table `commandes`
3. Créez une deuxième table `reporting` avec un agrégat par trimestre et région
4. Exécutez au moins 3 requêtes SQL directement depuis Python et affichez les résultats

In [71]:
import sqlite3
import os

# 1. creation de base SQLite ../output/ecommerce.db
db_path = os.path.join("output", "ecommerce.db")
conn = sqlite3.connect(db_path)  # connexion à la bdd

# 2. chargez le DataFrame dans une table `commandes`
commandes.to_sql("commandes", conn, if_exists="replace", index=False)
print("DataFrame chargé avec succès dans la base SQLite.")

# 3. creation d'une deuxième table `reporting` avec un agrégat par trimestre et région
req = """  DROP TABLE IF EXISTS reporting;
    CREATE TABLE reporting AS
    SELECT trimestre, region, SUM(montant_total) as ca
    FROM commandes
    GROUP BY trimestre, region
"""
curseur = conn.cursor()
curseur.executescript(req)


# 4. execution de 3 requetes sql
# req1 : CA par trimestre
req1 = """ SELECT
    trimestre,
    SUM(montant_total) AS ca
    FROM commandes
    GROUP BY trimestre;
"""
ca = pd.read_sql_query(req1,conn) 
print(ca)

# req2 : nb de commandes par statut
req2 = """ SELECT statut, COUNT(commande_id) as nb_commande
    FROM commandes
    GROUP BY statut
"""
nb_commande= pd.read_sql_query(req2,conn)
print(nb_commande)

# req3 : top 5 des commandes
req3 = """ SELECT *
    FROM commandes
    ORDER BY montant_total DESC
    LIMIT 5
"""
top_com = pd.read_sql_query(req3,conn)
print(top_com)

conn.close() 

DataFrame chargé avec succès dans la base SQLite.
         ca
0  15442.72
1  19068.97
      statut  nb_commande
0    annulée          602
1   en_cours          917
2    inconnu           62
3     livrée         4208
4  retournée          306
    commande_id client_id produit_id  quantite  prix_unitaire  montant_total  \
0  CMD-Q2-00469   CLI0124    PRD0088         9         793.25        7139.25   
1  CMD-Q2-00484   CLI0010    PRD0088         9         793.25        7139.25   
2  CMD-Q2-01525   CLI0367    PRD0088         9         793.25        7139.25   
3  CMD-Q2-01722   CLI0245    PRD0088         9         793.25        7139.25   
4  CMD-Q2-02763   CLI0335    PRD0088         9         793.25        7139.25   

         date_commande    statut mode_paiement region_livraison  ...  \
0  2024-05-08 00:00:00    livrée      Virement        Occitanie  ...   
1  2024-04-16 00:00:00    livrée            CB        Occitanie  ...   
2  2024-04-26 00:00:00  en_cours        PayPal             PA

### Ex L4 ⭐⭐⭐ — Parquet partitionné
Exportez le DataFrame en Parquet partitionné par `trimestre` et `region_livraison`.

Puis répondez :
- Combien de fichiers Parquet sont créés ?
- Quelle est la taille totale vs le Parquet non partitionné ?
- Quel est l'avantage du partitionnement pour PySpark ?

In [69]:
# Exporter le dataframe en parquet
commandes.to_parquet(
    "output/commandes_partitionne",
    partition_cols=["trimestre", "region_livraison"],
    index=False
)

# nb de fichiers Parquet
nb_fichier = 



In [67]:
commandes.columns

Index(['commande_id', 'client_id', 'produit_id', 'quantite', 'prix_unitaire',
       'montant_total', 'date_commande', 'statut', 'mode_paiement',
       'region_livraison', 'devise', 'source', 'annee', 'mois', 'trimestre',
       'jour_semaine', 'ville', 'region', 'segment', 'nom', 'categorie',
       'fournisseur', 'tranche_montant', 'saison', 'est_annulee', 'ca_perdu',
       'date_inscription', 'delai_inscription_commande'],
      dtype='str')

---
# 🔴 PIPELINE COMPLET

### Ex P1 ⭐⭐ — Pipeline modulaire
Écris 3 fonctions séparées :
- `extract(data_dir) -> dict`
- `transform(sources: dict) -> pd.DataFrame`
- `load(df, output_dir) -> None`

Chacune doit avoir une docstring et des logs.
Teste chaque fonction indépendamment.

In [ ]:
# TON CODE ICI


### Ex P2 ⭐⭐⭐ — Pipeline avec gestion d'erreurs
Améliore ton pipeline avec :
1. `try/except` sur chaque étape critique
2. Un log d'erreur détaillé si une étape échoue
3. La pipeline ne doit pas crasher complètement si un fichier est manquant — elle doit logger l'erreur et continuer avec les autres

In [ ]:
# TON CODE ICI


### Ex P3 ⭐⭐⭐ — Tests unitaires du pipeline
Écris 5 tests pour ton pipeline :
1. `test_extract_retourne_4_sources()`
2. `test_transform_supprime_doublons()`
3. `test_transform_pas_de_prix_negatif()`
4. `test_transform_montant_coherent()`
5. `test_load_cree_les_fichiers()`

Chaque test affiche ✅ PASS ou ❌ FAIL.

In [ ]:
# TON CODE ICI


### Ex P4 ⭐⭐⭐⭐ — PROJET FINAL : Pipeline automatisé
Crée un pipeline complet qui :

1. **Détecte automatiquement** tous les fichiers CSV dans `../data_etl/` qui commencent par `commandes_`
2. **Les traite tous** sans avoir à spécifier les chemins manuellement
3. **Génère un rapport HTML** avec :
   - Les statistiques clés (CA, nb commandes, taux annulation)
   - 3 graphiques Plotly intégrés
4. **Sauvegarde** en CSV + Parquet + SQLite
5. **Loggue** chaque étape avec timestamps

C'est exactement ce que vous ferez en PySpark la semaine prochaine, mais à l'échelle de plusieurs To.

📎 Docs utiles :
- [glob — trouver des fichiers](https://docs.python.org/3/library/glob.html)
- [logging](https://docs.python.org/3/library/logging.html)
- [Plotly — export HTML](https://plotly.com/python/interactive-html-export/)


In [ ]:
# TON CODE ICI
